In [ ]:
import pandas as pd
from tqdm import tqdm

In [ ]:
# read in the sample for class sessions as a basis for all information for class sessions
class_sessions_df = pd.read_parquet('Sample/sample_classes.parquet')
sample_aps_df = pd.read_parquet('Sample/sample_aps.parquet')
sample_classrooms_df = pd.read_parquet('Sample/sample_classrooms.parquet')


In [ ]:
#print(class_sessions_df.head())
print(class_sessions_df.dtypes)
print(sample_aps_df.dtypes)
print(sample_classrooms_df.dtypes)

In [ ]:
#Adding additional columns to the DataFrame
# Ensure string columns use Pandas' dedicated string dtype
class_sessions_df['group'] = pd.Series(dtype="string")

# Set numerical columns with nullable integer dtype (Int64)
class_sessions_df['number_aps'] = pd.Series(dtype="Int64")
class_sessions_df['number_radios'] = pd.Series(dtype="Int64")
class_sessions_df['number_beacons'] = pd.Series(dtype="Int64")
class_sessions_df['max_stations'] = pd.Series(dtype="Int64")
class_sessions_df['max_cu'] = pd.Series(dtype="Int64")
class_sessions_df['average_cu'] = pd.Series(dtype="Float64")
class_sessions_df['median_cu'] = pd.Series(dtype="Float64")
class_sessions_df['stations_enrolled'] = pd.Series(dtype="Float64")

# Compute class_duration in SECONDS (float since it involves time differences)
class_sessions_df['class_duration'] = (class_sessions_df['end_time'] - class_sessions_df['start_time']).dt.total_seconds().astype("Float64")

# Ensure duration_75 and duration_50 are also float types but nullable
class_sessions_df['duration_75'] = pd.Series(dtype="Float64")
class_sessions_df['duration_50'] = pd.Series(dtype="Float64")


In [ ]:

# Display the updated DataFrame
print(class_sessions_df.head())
print(class_sessions_df.dtypes)


In [ ]:
# Count occurrences of each Building-Room in sample_aps_df
aps_count = sample_aps_df['Building-Room'].value_counts()

# Map these counts to the number_aps column in class_sessions_df
class_sessions_df['number_aps'] = class_sessions_df['location'].map(aps_count).astype("Int64")


In [ ]:
#Group A, B, and C are from the sample selection based on classroom size
# Create a mapping from sample_classrooms_df: Location -> Group
location_to_group = sample_classrooms_df.set_index('Location')['Group']

# Map the Group values to class_sessions_df based on location
class_sessions_df['group'] = class_sessions_df['location'].map(location_to_group).astype("string")


In [ ]:
all_class_beacons_processed_df = pd.read_parquet('Data/all_class_beacons_processed.parquet')


In [ ]:
print(all_class_beacons_processed_df.head)

In [ ]:
# Populating more data in room_scount cu for room
# Iterate through class sessions
for idx, session in tqdm(class_sessions_df.iterrows(), total=len(class_sessions_df)):
    start = session['start_time']
    end = session['end_time']
    location = session['location']
    
    # Filter by time AND location
    mask = (
        (all_class_beacons_processed_df['aruba_erm.time'] >= start) &
        (all_class_beacons_processed_df['aruba_erm.time'] <= end) &
        (all_class_beacons_processed_df['location'] == location)
    )
    filtered = all_class_beacons_processed_df[mask]
    
    number_beacons = len(filtered)
    
    if number_beacons > 0:
        # Compute stats from room_scount
        max_scount = filtered['room_scount'].max()

        # Compute CU stats
        cu_values = filtered['wlan.qbss.cu'].dropna()
        max_cu = cu_values.max() if not cu_values.empty else 0
        avg_cu = cu_values.mean() if not cu_values.empty else 0
        median_cu = cu_values.median() if not cu_values.empty else 0
    else:
        max_scount = 0
        max_cu = avg_cu = median_cu = 0

    # Store values in the session row
    class_sessions_df.at[idx, 'max_stations'] = max_scount
    class_sessions_df.at[idx, 'stations_enrolled'] = max_scount / session['enrolled'] if session['enrolled'] else 0
    class_sessions_df.at[idx, 'max_cu'] = max_cu
    class_sessions_df.at[idx, 'average_cu'] = avg_cu
    class_sessions_df.at[idx, 'median_cu'] = median_cu
    class_sessions_df.at[idx, 'number_beacons'] = number_beacons



In [ ]:
print(class_sessions_df.head())

In [ ]:
# Calculate the total of max_stations and enrolled
total_max_stations = class_sessions_df['max_stations'].sum()
total_enrolled = class_sessions_df['enrolled'].sum()

# Prevent division by zero
if total_enrolled == 0:
    average_devices_per_enrolled = 0
else:
    average_devices_per_enrolled = total_max_stations / total_enrolled

print("Average devices per enrolled:", average_devices_per_enrolled)

In [ ]:
# populate the max scount from all_class_beacons_processed.parquet
# need to identify the APs that do not have any captures for the class and remove it from the sample